# LLMDTA — FNet Fourier Mixing: 3 cai tien re nhat (Tier 1) — Davis/Warm (Kaggle Notebook)

Notebook nay huan luyen va so sanh **4 bien the** tren cung 1 fold/seed de kiem tra nhanh (feasibility
probe) cac de xuat trong `IMPROVE_FNet_Proposal.md` (muc "Tier 1 - chi phi thap"):

| # | Ten bien the | Thay doi so voi FNet hien tai (`code/train_fnet.py`) |
|---|---|---|
| A | `FNet (hien tai)` | Khong doi — `FourierCrossMixing` khong tham so (mo lai lam moc so sanh) |
| B | `FNet + Learnable Filter` | Them 1 **filter phuc hoc duoc** nhan vao pho 2D truoc khi bien doi nguoc (GFNet-style, arXiv:2107.00645) |
| C | `FNet + Learnable Filter + k=5` | Nhu B, cong them doi `kernel_size` cua `Encoder` tu 7 -> **5** (finding tu DCI-SiteDTA, BMC Bioinformatics 2026) |
| D | `FNet + Learnable Filter + k=5 + GBA-Mixup` | Nhu C, cong them **GBA-Mixup-lite**: mixup giua cac cap drug-target *chung drug hoac chung protein* (tu MixingDTA, Bioinformatics 2025) thay vi mixup ngau nhien |

**Khong doi:** Encoder (Conv1d+GLU) cho bien the A/B, MoE head, optimizer, seed, fold, split du lieu — chi doi dung 1 thanh phan moi lan de so sanh cong bang, giong protocol da thong nhat trong `REPORT_Fourier_Mixing_DTA.md` §4.3.

**Chua lam trong notebook nay** (xem `IMPROVE_FNet_Proposal.md` de biet ly do va buoc tiep theo):
- Nang cap embedding ESM-C / MolFormer (Tier 1-(1)) — can chuan bi file pretrain rieng, xem `code_prepareEmb/_PreparePretrain_ESMC.ipynb`.
- AFNO-style mixing, Gated Fourier-Attention (Tier 2) — chi phi compute cao hon, de sau khi co tin hieu tu Tier 1.

## Truoc khi chay (bat buoc)

1. **Bat GPU**: panel ben phai -> *Settings* -> *Accelerator* -> chon **GPU T4 x2** (khuyen nghi — P100 co the bao loi "no kernel image available" voi phien ban PyTorch hien tai tren Kaggle).
2. **Bat Internet**: *Settings* -> *Internet* -> **On**.
3. **Add Data**: nut **+ Add Data** -> tim `llmdta` (chu so huu `christang0002`) -> **Add**.

Sau khi lam du 3 buoc tren, chon **Run All**. Thoi gian chay uoc tinh: ~4x thoi gian cua notebook FNet don le (`Kaggle_FNet_Davis_Warm.ipynb`) vi huan luyen 4 model tuan tu trong cung 1 session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Luu y: KHONG pin gensim==4.3.1 (khong co wheel dung san cho Python tren Kaggle
# -> pip phai build tu source va thuong loi). De pip tu chon ban gensim moi nhat.
!pip install -q rdkit gensim mol2vec wandb scikit-learn scipy tqdm
print('Da cai xong dependencies.')

In [ ]:
import os

REPO_URL = 'https://github.com/glucose20org/Temp.git'
REPO_BRANCH = 'quantum'
REPO_ROOT = '/kaggle/working/Temp'

if not os.path.exists(REPO_ROOT):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}
else:
    print('Repo da ton tai, bo qua clone.')

os.chdir(REPO_ROOT)
print('Working dir:', os.getcwd())

## Tim du lieu pretrain embedding tu dataset da Add Data

Cell duoi tu dong quet toan bo `/kaggle/input/**` de tim 2 file `*_drug_pretrain.pkl` va `*_esm_pretrain.pkl` cho dataset `davis`. Neu ten file trong dataset Kaggle khac pattern doan duoc, notebook se in ra cay thu muc `/kaggle/input` de ban tu xac dinh duong dan, roi gan thu cong vao `MANUAL_DRUG_PKL` / `MANUAL_PROT_PKL` o cell ke tiep.

In [ ]:
import glob, shutil, tarfile

DATASET = 'davis'         # davis | kiba | metz
RUNNING_SET = 'warm'       # warm | novel-drug | novel-prot | novel-pair

# Fold-data (train/valid/test csv) da co san trong repo duoi dang .tar.gz -> chi can giai nen
fold_root = os.path.join(REPO_ROOT, 'data', 'dta-5fold-dataset')
tar_path = os.path.join(fold_root, f'{DATASET}.tar.gz')
extracted_path = os.path.join(fold_root, DATASET)
if not os.path.exists(extracted_path) and os.path.exists(tar_path):
    print(f'Giai nen {tar_path} ...')
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(fold_root, filter='data')
        except TypeError:
            tf.extractall(fold_root)  # Python < 3.12 khong co tham so filter
print('Fold-data san sang tai:', extracted_path)

# Ghi de thu cong neu can (de trong '' de dung auto-detect)
MANUAL_DRUG_PKL = ''
MANUAL_PROT_PKL = ''

KAGGLE_INPUT_ROOT = '/kaggle/input'

def find_best_match(patterns, search_root):
    candidates = []
    for pat in patterns:
        candidates += glob.glob(os.path.join(search_root, '**', pat), recursive=True)
    return sorted(set(candidates))

target_dir = os.path.join(REPO_ROOT, 'data', DATASET)
os.makedirs(target_dir, exist_ok=True)
target_drug = os.path.join(target_dir, f'{DATASET}_drug_pretrain.pkl')
target_prot = os.path.join(target_dir, f'{DATASET}_esm_pretrain.pkl')

if MANUAL_DRUG_PKL:
    shutil.copy(MANUAL_DRUG_PKL, target_drug)
    print('Da copy (thu cong) drug pretrain ->', target_drug)
elif not os.path.exists(target_drug):
    drug_candidates = find_best_match([f'*{DATASET}*drug_pretrain*.pkl', f'*{DATASET}*mol2vec*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien drug pretrain:', drug_candidates)
    if drug_candidates:
        shutil.copy(drug_candidates[0], target_drug)
        print('Da copy drug pretrain ->', target_drug)

if MANUAL_PROT_PKL:
    shutil.copy(MANUAL_PROT_PKL, target_prot)
    print('Da copy (thu cong) prot pretrain ->', target_prot)
elif not os.path.exists(target_prot):
    prot_candidates = find_best_match([f'*{DATASET}*esm_pretrain*.pkl', f'*{DATASET}*esm*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien prot pretrain:', prot_candidates)
    if prot_candidates:
        shutil.copy(prot_candidates[0], target_prot)
        print('Da copy prot pretrain ->', target_prot)

if not os.path.exists(target_drug) or not os.path.exists(target_prot):
    print('\n[!] Khong tu dong tim thay du file pretrain. Cay thu muc /kaggle/input hien co:')
    for root, dirs, fs in os.walk(KAGGLE_INPUT_ROOT):
        depth = root.replace(KAGGLE_INPUT_ROOT, '').count(os.sep)
        if depth > 3:
            continue
        print('  ' * depth + os.path.basename(root) + '/')
        for f in fs[:15]:
            print('  ' * (depth + 1) + f)
    print('\nHay kiem tra duong dan chinh xac roi gan vao MANUAL_DRUG_PKL / MANUAL_PROT_PKL o tren va chay lai cell nay.')
    print('Neu chua Add Data: bam "+ Add Data" -> tim "llmdta" (christang0002/llmdta) -> Add.')
else:
    print('\nDa co du 2 file pretrain embedding cho', DATASET)

In [ ]:
import sys, copy, random, math, time
from collections import defaultdict
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

CODE_DIR = os.path.join(REPO_ROOT, 'code')
sys.path.insert(0, CODE_DIR)

from hyperparameter import HyperParameter
from LLMDTA import LLMDTA
from MyDataset import CustomDataSet, my_collate_fn
from train import cindex_score, regression_scores, load_pickle, set_seed, test as run_test

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (khong khuyen nghi cho notebook nay)')

FOLD = 0
EPOCHS = 40
BATCH_SIZE = 64
MAX_PATIENCE = 10
SEED = 0

hp = HyperParameter()
hp.set_dataset(DATASET)
hp.running_set = RUNNING_SET
hp.Epoch = EPOCHS
hp.Batch_size = BATCH_SIZE
hp.max_patience = MAX_PATIENCE

dataset_root = os.path.join(hp.data_root, hp.dataset, hp.running_set)
required_files = [hp.mol2vec_dir, hp.protvec_dir, hp.drugs_dir, hp.prots_dir]
for split in ['train', 'valid', 'test']:
    required_files.append(os.path.join(dataset_root, f'fold_{FOLD}_{split}.csv'))

missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError('Thieu file:\n' + '\n'.join(f'  - {m}' for m in missing))
print('Da co du file can thiet. San sang chay!')

In [ ]:
drug_df = pd.read_csv(hp.drugs_dir)
prot_df = pd.read_csv(hp.prots_dir)
mol2vec_dict = load_pickle(hp.mol2vec_dir)
protvec_dict = load_pickle(hp.protvec_dir)

train_dir = os.path.join(dataset_root, f'fold_{FOLD}_train.csv')
valid_dir = os.path.join(dataset_root, f'fold_{FOLD}_valid.csv')
test_dir  = os.path.join(dataset_root, f'fold_{FOLD}_test.csv')

train_df_raw = pd.read_csv(train_dir)
train_set = CustomDataSet(train_df_raw, hp)
valid_set = CustomDataSet(pd.read_csv(valid_dir), hp)
test_set  = CustomDataSet(pd.read_csv(test_dir), hp)

collate = lambda x: my_collate_fn(x, device, hp, drug_df, prot_df, mol2vec_dict, protvec_dict)
train_loader = DataLoader(train_set, batch_size=hp.Batch_size, shuffle=True, drop_last=True, collate_fn=collate)
valid_loader = DataLoader(valid_set, batch_size=hp.Batch_size, shuffle=False, drop_last=False, collate_fn=collate)
test_loader  = DataLoader(test_set, batch_size=hp.Batch_size, shuffle=False, drop_last=False, collate_fn=collate)

print(f'Train: {len(train_set)}  Valid: {len(valid_set)}  Test: {len(test_set)}')

## Bien the A — FNet hien tai (parameter-free, moc so sanh)

Giong het `code/train_fnet.py`: 2D DFT khong tham so (`torch.fft.fft` theo chieu hidden roi theo chieu
sequence, lay phan thuc), cong residual + LayerNorm.

In [ ]:
class FourierCrossMixing(nn.Module):
    """Parameter-free FNet-style Fourier mixing (giong het code/train_fnet.py)."""
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.out_ln = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        seq_q = query.shape[1]
        combined = torch.cat([query, key_value], dim=1)
        mixed = torch.fft.fft(torch.fft.fft(combined, dim=-1), dim=-2).real
        mixed_query = mixed[:, :seq_q, :]
        mixed_query = self.dropout(mixed_query)
        return self.out_ln(mixed_query + query)


class LLMDTA_FNet(LLMDTA):
    """Bien the A: LLMDTA (MoE) voi CrossAttention duoc thay bang FourierCrossMixing (khong tham so)."""
    def __init__(self, hp, device):
        super().__init__(hp, device)
        self.drug_cross_attn = FourierCrossMixing(self.hidden_dim, self.cross_attention_dropout)
        self.prot_cross_attn = FourierCrossMixing(self.hidden_dim, self.cross_attention_dropout)

## Bien the B — `LearnableFourierMixing` (GFNet-style, arXiv:2107.00645)

Khac bien the A o **duy nhat 1 diem**: sau khi bien doi 2D DFT (giong het A), nhan pho voi 1 **filter
phuc hoc duoc** `complex_weight` (shape `seq_len x hidden_dim`, khoi tao gan `(1, 0)` => hanh vi ~ giong
FNet goc luc bat dau train, roi hoc dan do lech). Day la "buoc ban tham so" ma `REPORT_Fourier_Mixing_DTA.md`
§6 da tu de xuat nhung chua cai dat. So luong tham so them vao duoc in ra khi khoi tao model (Cell chay Bien the B).

In [ ]:
class LearnableFourierMixing(nn.Module):
    """GFNet-style: FFT 2 chieu (hidden roi seq) + filter phuc hoc duoc tren mien tan so,
    roi IFFT 2 chieu, lay phan thuc. Khac FourierCrossMixing (bien the A) o cho co them
    self.complex_weight duoc nhan element-wise vao pho truoc buoc bien doi nguoc."""
    def __init__(self, hidden_dim, seq_len, dropout=0.1):
        super().__init__()
        # Khoi tao gan (re=1, im=0) => filter ~ identity luc bat dau train (on dinh hon random init)
        weight = torch.zeros(seq_len, hidden_dim, 2)
        weight[..., 0] = 1.0
        weight = weight + torch.randn_like(weight) * 0.02
        self.complex_weight = nn.Parameter(weight)
        self.out_ln = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        seq_q = query.shape[1]
        combined = torch.cat([query, key_value], dim=1)  # (b, L, d)
        L = combined.shape[1]
        x = torch.fft.fft(combined, dim=-1)   # mix theo hidden dim (giong FNet goc)
        x = torch.fft.fft(x, dim=-2)          # mix theo sequence dim (giong FNet goc)
        weight = torch.view_as_complex(self.complex_weight[:L])   # (L, d) complex
        x = x * weight.unsqueeze(0)           # <-- diem khac biet voi bien the A: filter hoc duoc
        x = torch.fft.ifft(x, dim=-2)
        x = torch.fft.ifft(x, dim=-1)
        mixed = x.real
        mixed_query = self.dropout(mixed[:, :seq_q, :])
        return self.out_ln(mixed_query + query)


class LLMDTA_FNet_Learnable(LLMDTA):
    """Bien the B: nhu A nhung dung LearnableFourierMixing thay FourierCrossMixing."""
    def __init__(self, hp, device):
        super().__init__(hp, device)
        seq_len = hp.substructure_max_len + hp.prot_max_len
        self.drug_cross_attn = LearnableFourierMixing(self.hidden_dim, seq_len, self.cross_attention_dropout)
        self.prot_cross_attn = LearnableFourierMixing(self.hidden_dim, seq_len, self.cross_attention_dropout)

## Bien the C — cong them `kernel_size=5` cho Encoder (DCI-SiteDTA finding)

`code/LLMDTA.py` dang co dinh `self.kernel_size = 7` trong `Encoder`. DCI-SiteDTA (BMC Bioinformatics 2026)
sweep k trong {9,7,5,3} va tim ra **k=5** toi uu tren ca Davis va KIBA cho ca binding-site detection lan
affinity regression. `EncoderK` ben duoi giong het `Encoder` goc, chi them tham so `kernel_size`.

In [ ]:
class EncoderK(nn.Module):
    """Giong het class Encoder trong code/LLMDTA.py, chi them tham so kernel_size (mac dinh goc = 7)."""
    def __init__(self, max_len, input_dim, device, dropout=0.1, hidden_dim=128, kernel_size=5):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.do = nn.Dropout(dropout)
        self.register_buffer('scale', torch.sqrt(torch.FloatTensor([0.5])))
        self.input_norm = nn.LayerNorm(self.input_dim)
        self.fc = nn.Linear(self.input_dim, self.hidden_dim)
        self.ln = nn.LayerNorm(self.hidden_dim)
        pad = (self.kernel_size - 1) // 2
        self.convs = nn.ModuleList([
            nn.Conv1d(self.hidden_dim, self.hidden_dim * 2, self.kernel_size, padding=pad)
            for _ in range(3)
        ])
        self.max_pool = nn.MaxPool1d(max_len)

    def forward(self, feat_map):
        feat_map = self.input_norm(feat_map)
        h_map = self.fc(feat_map)
        h_map = h_map.permute(0, 2, 1)
        for conv in self.convs:
            conved = conv(self.do(h_map))
            conved = F.glu(conved, dim=1)
            conved = (conved + h_map) * self.scale
            h_map = conved
        pool_map = self.max_pool(h_map).squeeze(-1)
        h_map = h_map.permute(0, 2, 1)
        h_map = self.ln(h_map)
        return h_map, pool_map


class LLMDTA_FNet_Learnable_K5(LLMDTA_FNet_Learnable):
    """Bien the C: nhu B, cong them Encoder voi kernel_size=5 thay vi 7."""
    def __init__(self, hp, device, kernel_size=5):
        super().__init__(hp, device)
        self.drug_embed = EncoderK(hp.drug_max_len, self.mol2vec_dim, device, self.encoder_dropout, kernel_size=kernel_size)
        self.prot_embed = EncoderK(hp.prot_max_len, self.protvec_dim, device, self.encoder_dropout, kernel_size=kernel_size)

## Bien the D — cong them GBA-Mixup-lite (MixingDTA-style augmentation)

Khac 3 bien the tren o **du lieu huan luyen**, khong dong den kien truc (dung lai chinh
`LLMDTA_FNet_Learnable_K5` cua bien the C). `GBAMixupDataSet` cho moi anchor pair, voi xac suat
`mixup_prob`, tim 1 "hang xom" trong tap train **chung drug_id hoac chung prot_id** (dung nguyen ly
"guilt-by-association" cua MixingDTA — cac thuc the lien quan co xu huong chia se chuc nang/ai luc),
roi noi suy embedding + nhan theo he so `lambda ~ Beta(alpha, alpha)`. Khac ban goc GBA-Mixup (dung ca
6 kich ban neighbor + meta-predictor rieng), day la ban **rut gon 1 kich ban** ("protein-or-drug") de
co the chay ngay trong 1 notebook — du de kiem tra tin hieu ban dau truoc khi dau tu ban day du.

In [ ]:
class GBAMixupDataSet(Dataset):
    """Mixup giua cac cap D-T chung drug hoac chung protein (GBA-Mixup-lite, MixingDTA 2025)."""
    def __init__(self, df, alpha=0.4, mixup_prob=0.5):
        self.df = df.reset_index(drop=True)
        self.alpha = alpha
        self.mixup_prob = mixup_prob
        # Dung vi tri cot 0/1 (drug_id/prot_id) thay vi ten cot, khop cach MyDataset.py doc du lieu
        drug_col, prot_col = self.df.columns[0], self.df.columns[1]
        self.drug_ids = self.df[drug_col].values
        self.prot_ids = self.df[prot_col].values
        self.by_drug = defaultdict(list)
        self.by_prot = defaultdict(list)
        for idx in range(len(self.df)):
            self.by_drug[self.drug_ids[idx]].append(idx)
            self.by_prot[self.prot_ids[idx]].append(idx)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        anchor = self.df.iloc[index, :]
        candidates = set()
        if np.random.rand() < self.mixup_prob:
            candidates = set(self.by_drug[self.drug_ids[index]]) | set(self.by_prot[self.prot_ids[index]])
            candidates.discard(index)
        if candidates:
            nb_idx = int(np.random.choice(list(candidates)))
            neighbor = self.df.iloc[nb_idx, :]
            lam = float(np.random.beta(self.alpha, self.alpha))
        else:
            neighbor = anchor
            lam = 1.0
        return anchor, neighbor, lam


def gba_mixup_collate_fn(batch_data, device, hp, drug_df, prot_df, mol2vec_dict, protvec_dict):
    anchors = [b[0] for b in batch_data]
    neighbors = [b[1] for b in batch_data]
    lams = torch.tensor([b[2] for b in batch_data], dtype=torch.float32, device=device)

    a_drug_vec, a_prot_vec, a_drug_mat, a_drug_mask, a_prot_mat, a_prot_mask, a_label = my_collate_fn(
        anchors, device, hp, drug_df, prot_df, mol2vec_dict, protvec_dict)
    n_drug_vec, n_prot_vec, n_drug_mat, n_drug_mask, n_prot_mat, n_prot_mask, n_label = my_collate_fn(
        neighbors, device, hp, drug_df, prot_df, mol2vec_dict, protvec_dict)

    lam1 = lams.view(-1, 1)          # de nhan voi tensor (batch, dim)
    lam2 = lams.view(-1, 1, 1)       # de nhan voi tensor (batch, seq, dim)

    drug_vec = lam1 * a_drug_vec + (1 - lam1) * n_drug_vec
    prot_vec = lam1 * a_prot_vec + (1 - lam1) * n_prot_vec
    drug_mat = lam2 * a_drug_mat + (1 - lam2) * n_drug_mat
    prot_mat = lam2 * a_prot_mat + (1 - lam2) * n_prot_mat
    drug_mask = torch.maximum(a_drug_mask, n_drug_mask)
    prot_mask = torch.maximum(a_prot_mask, n_prot_mask)
    label = lams * a_label + (1 - lams) * n_label

    return drug_vec, prot_vec, drug_mat, drug_mask, prot_mat, prot_mask, label


mixup_dataset = GBAMixupDataSet(train_df_raw, alpha=0.4, mixup_prob=0.5)
mixup_collate = lambda x: gba_mixup_collate_fn(x, device, hp, drug_df, prot_df, mol2vec_dict, protvec_dict)
mixup_train_loader = DataLoader(mixup_dataset, batch_size=hp.Batch_size, shuffle=True, drop_last=True, collate_fn=mixup_collate)

print(f'GBA-Mixup dataset san sang: {len(mixup_dataset)} anchor pairs, mixup_prob=0.5, alpha=0.4')

In [ ]:
# ---------------------------------------------------------------
# Vong lap train/eval dung chung cho ca 4 bien the (chi doi model_cls + train_loader)
# ---------------------------------------------------------------

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run_training(model_cls, run_name, hp, train_loader, valid_loader, test_loader, device, seed=0, verbose=True, model_kwargs=None):
    set_seed(seed)
    model_kwargs = model_kwargs or {}
    model = model_cls(hp, device, **model_kwargs).to(device)
    n_params = count_params(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=hp.Learning_rate, betas=(0.9, 0.999), weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp.Epoch, eta_min=1e-6)
    criterion = F.mse_loss

    history = {'epoch': [], 'train_mse': [], 'train_ci': [], 'valid_mse': [], 'valid_ci': []}
    best_valid_mse = float('inf')
    best_state = None
    patience = 0

    t0 = time.time()
    for epoch in range(1, hp.Epoch + 1):
        if hasattr(model, 'reset_usage_stats'):
            model.reset_usage_stats()

        model.train()
        preds, labels = [], []
        for batch_data in train_loader:
            mol_vec, prot_vec, mol_mat, mol_mat_mask, prot_mat, prot_mat_mask, affinity = batch_data
            predictions, gate_info = model(mol_vec, mol_mat, mol_mat_mask, prot_vec, prot_mat, prot_mat_mask, return_gate_info=True)
            preds += predictions.detach().cpu().numpy().reshape(-1).tolist()
            labels += affinity.detach().cpu().numpy().reshape(-1).tolist()

            loss = criterion(predictions.squeeze(), affinity)

            if gate_info.get('gate_weights') is not None and hasattr(model, 'compute_load_balance_loss'):
                lb_loss = model.compute_load_balance_loss(gate_info['gate_weights'])
                total_loss = loss + hp.load_balance_weight * lb_loss
            else:
                total_loss = loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        train_mse, _, train_ci, _, _, _ = regression_scores(np.array(labels), np.array(preds), is_valid=False)
        valid_mse, valid_rmse, _, valid_r2, _, _ = run_test(model, valid_loader, is_valid=True)
        scheduler.step()

        history['epoch'].append(epoch)
        history['train_mse'].append(train_mse)
        history['train_ci'].append(train_ci)
        history['valid_mse'].append(valid_mse)

        if valid_mse < best_valid_mse:
            best_valid_mse = valid_mse
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1

        if verbose and (epoch % 5 == 0 or epoch == 1 or patience > hp.max_patience):
            print(f'[{run_name}] epoch {epoch:3d}/{hp.Epoch}  train_mse={train_mse:.4f} train_ci={train_ci:.4f}  '
                  f'valid_mse={valid_mse:.4f}  best_valid_mse={best_valid_mse:.4f}  patience={patience}')

        if patience > hp.max_patience:
            print(f'[{run_name}] early stopping at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_mse, test_rmse, test_ci, test_r2, test_pearson, test_spearman = run_test(model, test_loader, is_valid=False)
    elapsed = time.time() - t0

    result = {
        'name': run_name,
        'n_params': n_params,
        'epochs_ran': history['epoch'][-1],
        'elapsed_sec': elapsed,
        'test_mse': test_mse, 'test_rmse': test_rmse, 'test_ci': test_ci,
        'test_r2': test_r2, 'test_pearson': test_pearson, 'test_spearman': test_spearman,
    }
    print(f'[{run_name}] TEST  mse={test_mse:.4f}  rmse={test_rmse:.4f}  ci={test_ci:.4f}  '
          f'r2={test_r2:.4f}  pearson={test_pearson:.4f}  ({n_params:,} params, {elapsed:.1f}s)')
    return model, history, result

## Chay Bien the A — FNet hien tai (moc so sanh)

In [ ]:
model_a, history_a, result_a = run_training(
    LLMDTA_FNet, 'A: FNet (hien tai)', hp, train_loader, valid_loader, test_loader, device, seed=SEED
)

## Chay Bien the B — FNet + Learnable Fourier Filter (GFNet-style)

In [ ]:
model_b, history_b, result_b = run_training(
    LLMDTA_FNet_Learnable, 'B: FNet + Learnable Filter', hp, train_loader, valid_loader, test_loader, device, seed=SEED
)
print(f"So tham so them vao so voi A: {result_b['n_params'] - result_a['n_params']:,}")

## Chay Bien the C — B + Encoder kernel_size=5

In [ ]:
model_c, history_c, result_c = run_training(
    LLMDTA_FNet_Learnable_K5, 'C: B + kernel_size=5', hp, train_loader, valid_loader, test_loader, device, seed=SEED,
    model_kwargs={'kernel_size': 5}
)

## Chay Bien the D — C + GBA-Mixup (chi doi train_loader, giu nguyen kien truc cua C)

In [ ]:
model_d, history_d, result_d = run_training(
    LLMDTA_FNet_Learnable_K5, 'D: C + GBA-Mixup', hp, mixup_train_loader, valid_loader, test_loader, device, seed=SEED,
    model_kwargs={'kernel_size': 5}
)

## Bang so sanh 4 bien the

In [ ]:
import json

os.makedirs('/kaggle/working/results', exist_ok=True)

all_results = [result_a, result_b, result_c, result_d]
results_df = pd.DataFrame(all_results)[
    ['name', 'test_mse', 'test_rmse', 'test_ci', 'test_r2', 'test_pearson', 'n_params', 'epochs_ran', 'elapsed_sec']
]
results_df = results_df.sort_values('test_ci', ascending=False).reset_index(drop=True)

with pd.option_context('display.float_format', '{:.4f}'.format):
    print(results_df.to_string(index=False))

results_df.to_csv('/kaggle/working/results/comparison_tier1.csv', index=False)
with open('/kaggle/working/results/comparison_tier1.json', 'w') as f:
    json.dump(all_results, f, indent=2)

for h, tag in [(history_a, 'a_fnet'), (history_b, 'b_learnable'), (history_c, 'c_k5'), (history_d, 'd_mixup')]:
    pd.DataFrame(h).to_csv(f'/kaggle/working/results/history_{tag}.csv', index=False)

print('\nDa luu: /kaggle/working/results/comparison_tier1.csv (+ .json) va 4 file history_*.csv')
print('Nho vao tab "Output" cua Kaggle de tai ve sau khi notebook chay xong.')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
names = results_df['name'].tolist()

ax[0].bar(names, results_df['test_ci'], color=colors)
ax[0].set_title('Test CI (cao hon = tot hon)')
ax[0].tick_params(axis='x', rotation=25)

ax[1].bar(names, results_df['test_mse'], color=colors)
ax[1].set_title('Test MSE (thap hon = tot hon)')
ax[1].tick_params(axis='x', rotation=25)

for h, tag, c in [(history_a, 'A', colors[0]), (history_b, 'B', colors[1]), (history_c, 'C', colors[2]), (history_d, 'D', colors[3])]:
    ax[2].plot(h['epoch'], h['valid_mse'], label=tag, color=c)
ax[2].set_title('Valid MSE theo epoch')
ax[2].set_xlabel('Epoch')
ax[2].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/results/comparison_tier1.png', dpi=150)
plt.show()

## Ket qua & buoc tiep theo

**Canh bao quan trong:** ket qua tren la **1 fold (fold 0), 1 seed** — chi la tin hieu nhanh (feasibility
probe), **chua** du de ket luan bien the nao tot hon that su. Truoc khi dua vao paper/report, phai lam
theo dung protocol da thong nhat trong `REPORT_Fourier_Mixing_DTA.md` §4.3:

1. Chay lai voi **5-fold x >=3 seed** cho (it nhat) bien the thang trong bang tren + bien the A (moc so sanh).
2. **Paired t-test** (`scipy.stats.ttest_rel`) tren CI giua tung cap bien the, cung fold + seed.
3. Neu bien the B/C/D thang co y nghia thong ke tren `davis/warm`, mo rong sang `novel-drug/novel-prot/novel-pair`
   (muc tieu cold-start chinh cua LLMDTA) va `kiba`/`metz` de kiem tra tinh tong quat.
4. Xem xet **nang cap embedding ESM-C/MolFormer** (Tier 1-(1) trong `IMPROVE_FNet_Proposal.md`) — theo MixingDTA day co the la don bay lon nhat, doc lap voi 3 cai tien da thu trong notebook nay.

Toan bo phan tich va cac de xuat khac (Tier 2/3, ly do lua chon, tham chieu paper) nam trong
`IMPROVE_FNet_Proposal.md` o thu muc goc repo.